# RAG Pipeline — Retrieval-Augmented Generation

**Phase 03 · Core Notebook**

---

## ⚠️ Prerequisites

| Requirement | How to satisfy |
|---|---|
| **Ollama running** | `ollama serve` in a terminal |
| **llama3 pulled** | `ollama pull llama3` |
| **Python packages** | `pip install langchain langchain-community chromadb sentence-transformers pandas faker` |

All inference is **100% local** — no API keys, no cloud costs.

---

**By the end of this notebook you will be able to:**

- Explain why LLMs hallucinate and what RAG does to fix it
- Chunk documents with LangChain and store them in ChromaDB
- Retrieve relevant chunks by semantic similarity
- Build a complete `rag_answer()` function that grounds answers in your data
- Identify and explain the three main RAG failure modes
- Extend RAG with conversation history for multi-turn dialogue

---

## Section 1 — The Problem RAG Solves

### Why LLMs hallucinate

A language model is a **probabilistic next-token predictor** trained on a fixed
snapshot of the internet. It has no access to:

- Your private documents
- Events after its training cutoff
- Real-time data

When asked a question it cannot answer from training data, it does not say
"I don't know" — it generates the *most likely-sounding* continuation of the
prompt. This produces fluent, confident, **wrong** answers. This is hallucination.

### What grounding means

**Grounding** means anchoring the model's output to a specific, verifiable source
of truth. Instead of relying on what the model *memorised*, you *provide* the
relevant facts in the prompt. The model's job shifts from "recall" to
"comprehension and summarisation" — tasks it is much better at.

### The RAG flow

```
┌─────────────────────────────────────────────────────────────────────┐
│                        RAG PIPELINE                                 │
│                                                                     │
│  INDEXING (done once)                                               │
│  ──────────────────────────────────                                 │
│  Raw documents                                                      │
│       │                                                             │
│       ▼                                                             │
│  Chunker  ──▶  [chunk₁] [chunk₂] [chunk₃] ... [chunkₙ]            │
│                     │                                              │
│                     ▼                                              │
│              Embedding model  ──▶  vectors                         │
│                     │                                              │
│                     ▼                                              │
│              ChromaDB (local vector store)                         │
│                                                                     │
│  QUERYING (done per question)                                       │
│  ──────────────────────────────────                                 │
│  User question                                                      │
│       │                                                             │
│       ▼                                                             │
│  Embed question  ──▶  similarity search  ──▶  top-k chunks         │
│                                                    │               │
│                                                    ▼               │
│                             Prompt template (context + question)   │
│                                                    │               │
│                                                    ▼               │
│                                          Ollama (local LLM)        │
│                                                    │               │
│                                                    ▼               │
│                                         Grounded answer ✓          │
└─────────────────────────────────────────────────────────────────────┘
```

### Key insight

RAG is not about making the model smarter. It is about giving the model
the **right information at the right time**, so it can do what it is good
at: reading, summarising, and reasoning over text.

---

## Section 2 — Load and Chunk Documents

### Why chunking matters

A vector store works by comparing the **embedding of your query** to the
**embedding of each stored piece of text**. If you store whole documents
(thousands of words), two problems arise:

1. **Dilution** — one embedding must represent the entire document. Specific
   facts get drowned out by surrounding text.
2. **Context overflow** — the retrieved document may be too long to fit in
   the LLM's context window.

Chunking splits documents into smaller pieces so each chunk is semantically
focused and fits cleanly in context.

### Choosing `chunk_size` and `overlap`

| Parameter | Typical value | Effect |
|-----------|--------------|--------|
| `chunk_size` | 200–500 chars | Smaller = more precise retrieval; larger = more context per chunk |
| `chunk_overlap` | 10–20% of chunk_size | Prevents answers at chunk boundaries being split across two chunks |

**`RecursiveCharacterTextSplitter`** tries to split on paragraph breaks `\n\n`,
then sentence breaks `\n`, then spaces — it prefers natural boundaries over
arbitrary character positions.

### What to look for
- Each chunk is ≤ 300 characters
- The overlap region appears at both the end of one chunk and the start of the next

In [ ]:
import os
import random
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter

NOTEBOOK_DIR = os.path.abspath('')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, '..', '..', 'data', 'raw')
TICKETS_PATH = os.path.join(DATA_DIR, 'support_tickets.csv')

# ── Auto-generate support_tickets.csv if missing ──────────────────────────
if not os.path.exists(TICKETS_PATH):
    print('support_tickets.csv not found — generating...')
    try:
        from faker import Faker
        fake = Faker()
    except ImportError:
        raise SystemExit('Run: pip install faker')

    HIGH = [
        ('Production database is down',
         'Our entire production database cluster has been unreachable for 30 minutes. '
         'All customers are affected. Revenue impact is severe. '
         'Engineers are on-call but need platform access restored urgently.'),
        ('Critical security breach detected',
         'We have detected unauthorised access to customer PII. '
         'The attack appears ongoing. Immediate escalation required. '
         'All API keys for the affected service should be rotated immediately.'),
        ('Payment processing completely broken',
         'No payments are going through. Checkout is broken for 100% of users. '
         'We are losing revenue every minute. '
         'Last deployment was 2 hours ago and rollback has not fixed the issue.'),
        ('SSL certificate expired — site unreachable',
         'Our SSL cert expired 2 hours ago. All users see a browser security warning. '
         'The site is effectively offline. Auto-renewal did not trigger.'),
        ('Authentication service returning 500 errors',
         'Users cannot log in. Auth service has been throwing 500s since the '
         'last deployment 45 minutes ago. The rollback procedure has not resolved it.'),
        ('Data corruption in billing records',
         'Invoice amounts are incorrect for approximately 2,000 customers. '
         'The monthly billing cycle runs in 6 hours. '
         'Root cause appears to be a bad migration script applied yesterday.'),
        ('All backups missing from last 7 days',
         'Scheduled backup job silently failed for a week. '
         'We have no valid recovery point. Risk of permanent data loss is high.'),
        ('API rate limits not enforced — data leak risk',
         'Our public API has stopped enforcing rate limits. '
         'Malicious actors are actively scraping all customer data at high volume.'),
    ]
    MEDIUM = [
        ('Dashboard export generates wrong totals',
         'The CSV export from the analytics dashboard shows figures that do not '
         'match the on-screen numbers. Discrepancies are around 5%. '
         'Affects finance team monthly reporting.'),
        ('Email notifications delayed by 4+ hours',
         'Users report that welcome and password-reset emails arrive hours late. '
         'This is causing increased support calls from users locked out of accounts.'),
        ('Search returns irrelevant results after update',
         'Since last Monday\'s release, the search bar frequently returns unrelated '
         'items. A query for "invoice" returns blog posts. Users are complaining.'),
        ('Mobile app crashes on iOS 17.4',
         'Several users report the iOS app crashes immediately on launch after '
         'upgrading to iOS 17.4. The issue is reproducible. Android is unaffected.'),
        ('Charts not rendering in Firefox',
         'All bar and line charts are blank in Firefox 124+. '
         'Charts render correctly in Chrome and Safari. Affects roughly 15% of users.'),
        ('Bulk import fails on files larger than 5 MB',
         'The CSV import wizard silently fails for files over 5 MB '
         'without showing an error. Users lose work without knowing why.'),
        ('Report generation taking over 10 minutes',
         'Monthly summary reports that used to generate in 30 seconds '
         'now take over 10 minutes. Database query plan may have degraded.'),
        ('Wrong timezone applied to scheduled tasks',
         'Scheduled tasks run in UTC rather than the configured user timezone. '
         'This causes tasks to fire 1-8 hours late depending on user location.'),
    ]
    LOW = [
        ('Request to add dark mode',
         'Many users have requested a dark mode theme option. '
         'Would be nice to have for evening use. No business impact.'),
        ('Typo on pricing page',
         'The word "anually" should be "annually" on the /pricing page. Minor issue.'),
        ('Tooltip text is hard to read',
         'The tooltip on the settings page uses light grey text on white background. '
         'Accessibility concern but low impact overall.'),
        ('Add keyboard shortcut for save',
         'It would be helpful to have Ctrl+S / Cmd+S save the current form. '
         'Power users would appreciate this quality-of-life improvement.'),
        ('Docs link broken on help page',
         'The Getting Started link in the Help section returns a 404. '
         'Should be fixed in the next content deploy.'),
        ('Export button should confirm before running',
         'Users accidentally trigger large exports. '
         'A confirmation dialog would prevent accidental heavy jobs.'),
        ('Date picker does not support manual entry',
         'The date picker widget requires clicking. '
         'Users who prefer typing cannot enter dates directly.'),
        ('Add option to change default currency',
         'The UI always defaults to USD. '
         'Allow users to set a default display currency in profile settings.'),
    ]

    random.seed(42)
    rows = []
    ticket_id = 1001
    for items, priority in [(HIGH, 'high'), (MEDIUM, 'medium'), (LOW, 'low')]:
        for subject, body in items * 5:
            rows.append({'id': ticket_id, 'subject': subject,
                         'body': body, 'priority': priority,
                         'created_by': fake.name()})
            ticket_id += 1
    random.shuffle(rows)
    os.makedirs(DATA_DIR, exist_ok=True)
    pd.DataFrame(rows).to_csv(TICKETS_PATH, index=False)
    print(f'  Generated {len(rows)} tickets → {TICKETS_PATH}')

# ── Load tickets ──────────────────────────────────────────────────────────
df = pd.read_csv(TICKETS_PATH)
print(f'Loaded {len(df)} tickets | priority distribution:')
print(df['priority'].value_counts().to_dict())

# ── Build raw documents (subject + body concatenated) ─────────────────────
docs_raw = [
    f"[Ticket {row['id']} | Priority: {row['priority'].upper()}]\n"
    f"Subject: {row['subject']}\n"
    f"Body: {row['body']}"
    for _, row in df.iterrows()
]
print(f'\nExample raw document (first ticket):')
print('─' * 60)
print(docs_raw[0])

# ── Chunk with RecursiveCharacterTextSplitter ─────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

all_chunks = splitter.create_documents(
    texts=docs_raw,
    metadatas=[{'ticket_id': int(row['id']), 'priority': row['priority']}
               for _, row in df.iterrows()],
)

print(f'\nChunking results:')
print(f'  Input documents : {len(docs_raw)}')
print(f'  Output chunks   : {len(all_chunks)}')
print(f'  Avg chunk length: {sum(len(c.page_content) for c in all_chunks) / len(all_chunks):.0f} chars')

print(f'\nFirst chunk:')
print('─' * 60)
print(all_chunks[0].page_content)
print(f'\nSecond chunk (note overlap with first):')
print('─' * 60)
print(all_chunks[1].page_content)

---

## Section 3 — Build a Vector Store with Chroma

### Vector store vs regular database

| | SQL / NoSQL DB | Vector Store (Chroma) |
|---|---|---|
| **Query type** | Exact match (`WHERE id = 42`) | Semantic similarity (`nearest neighbours to query vector`) |
| **Index type** | B-tree, hash | HNSW (approximate nearest neighbour) |
| **Storage** | Rows / documents | Vectors (float arrays) + metadata |
| **Use case** | Structured lookups | "Find text that means the same thing as this query" |

A regular database cannot answer the query *"find support tickets similar to
'users cannot authenticate'"* without exact keyword matching (which misses
synonyms like "login broken", "sign-in failing", "access denied").
A vector store answers it naturally.

### How Chroma works locally

1. You pass text + metadata to `Chroma.from_documents()`
2. Chroma calls your embedding function on each chunk → float vector
3. Vectors are indexed with HNSW in a local SQLite file (`./chroma_db/`)
4. At query time: embed query → approximate nearest-neighbour search → return top-k chunks

No server required. Everything lives on disk in `./chroma_db/`.

### Embedding model

We use **`all-MiniLM-L6-v2`** from `sentence-transformers` (22 MB, runs on CPU,
free, no API key). The same model used in Phase 01.

### What to look for
- Collection size should equal the number of chunks from Section 2
- Embeddings are 384-dimensional float32 arrays

In [ ]:
import shutil
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

CHROMA_DIR = os.path.join(NOTEBOOK_DIR, 'chroma_db')

# Fresh build each run (remove old db if present)
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
    print('Removed existing chroma_db')

print('Loading embedding model (all-MiniLM-L6-v2)...')
embedding_fn = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print(f'Embedding {len(all_chunks)} chunks and indexing in Chroma...')
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_fn,
    persist_directory=CHROMA_DIR,
)

collection_size = vectorstore._collection.count()
print(f'\nVector store ready')
print(f'  Persist directory : {CHROMA_DIR}')
print(f'  Collection size   : {collection_size} chunks')
print(f'  Embedding dims    : 384')

# Spot-check: what does one raw embedding look like?
sample_vec = embedding_fn.embed_query('test')
print(f'\nSample query embedding (first 8 of 384 floats):')
print([round(v, 4) for v in sample_vec[:8]])

---

## Section 4 — Retrieval

### Similarity search and the role of k

When a query arrives:

1. The query is embedded with the **same model** used to index chunks
2. Chroma computes **cosine similarity** between the query vector and all stored vectors
3. The **top-k** most similar chunks are returned

**Why retrieve more than you need?**

Semantic similarity is imperfect — a highly similar chunk may not contain the
exact answer, while the true answer may be in the 4th or 5th result. The common
pattern is to retrieve `k=5` or `k=10` and then either:

- Pass all of them to the LLM (let the model pick what is relevant)
- Re-rank them with a cross-encoder (Phase 04)
- Filter by a minimum similarity threshold

### Similarity scores

Chroma returns **L2 distance** (lower = more similar) when using
`similarity_search_with_score`. We convert to a 0–1 score with
`score = 1 / (1 + distance)` so that 1.0 = perfect match.

### What to look for
- High-priority queries should retrieve high-priority ticket chunks
- Scores drop off quickly after the top 1–2 results
- Notice how semantically equivalent phrases ("can't log in", "authentication broken") retrieve the same chunks

In [ ]:
def retrieve(query: str, k: int = 3) -> list[tuple]:
    """Return top-k (chunk, score) tuples for a query."""
    results = vectorstore.similarity_search_with_score(query, k=k)
    # Convert L2 distance → similarity score (0 to 1)
    return [(doc, round(1 / (1 + dist), 3)) for doc, dist in results]


test_queries = [
    'Users cannot log in to the platform',
    'Payment checkout is failing',
    'The site is showing a security warning',
    'Charts are not displaying correctly',
    'Request for a new feature to improve usability',
]

for query in test_queries:
    print(f'\nQuery: "{query}"')
    print('─' * 60)
    hits = retrieve(query, k=3)
    for rank, (doc, score) in enumerate(hits, 1):
        priority  = doc.metadata.get('priority', '?').upper()
        ticket_id = doc.metadata.get('ticket_id', '?')
        snippet   = doc.page_content[:120].replace('\n', ' ')
        print(f'  [{rank}] score={score:.3f} | {priority:6} | ticket={ticket_id} | {snippet}...')

---

## Section 5 — Augmented Generation (Full RAG)

### The RAG prompt template

The final ingredient is a **prompt that combines retrieved context with the
user's question**. The template looks like:

```
You are a support assistant. Answer ONLY using the context below.
If the answer is not in the context, say "I don't have that information."

CONTEXT:
{chunk_1}
{chunk_2}
{chunk_3}

QUESTION: {user_question}

ANSWER:
```

### Why "only use the provided context"?

Without this instruction, the model blends retrieved context with its own
training knowledge. The result feels helpful but may contain hallucinated
details not in your documents. The instruction forces the model to stay
grounded — if it cannot answer from context, it says so.

### What to look for
- Answers should reference specific ticket details (IDs, subjects) that only exist in your CSV
- "I don't have that information" is a correct and desirable response when the data lacks the answer
- Speed: each call embeds the query + sends ~600 tokens to Ollama, so expect 5–30 seconds per question

In [ ]:
import requests
import textwrap

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL      = 'llama3'

RAG_SYSTEM_PROMPT = (
    'You are a helpful support desk assistant. '
    'Answer the question using ONLY the context provided below. '
    'If the answer cannot be found in the context, '
    'respond with exactly: "I don\'t have that information." '
    'Do not use your general knowledge. Be concise — 2 to 4 sentences maximum.'
)


def ollama_chat(messages: list[dict], temperature: float = 0.1) -> str:
    """Send messages to Ollama; return assistant reply string."""
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={'model': MODEL, 'messages': messages,
                  'stream': False, 'options': {'temperature': temperature}},
            timeout=120,
        )
        resp.raise_for_status()
    except requests.exceptions.ConnectionError:
        return '[ERROR] Cannot reach Ollama. Ensure `ollama serve` is running.'
    return resp.json()['message']['content']


def format_context(chunks: list[tuple]) -> str:
    """Format retrieved chunks into a numbered context block."""
    parts = []
    for i, (doc, score) in enumerate(chunks, 1):
        parts.append(f'[Source {i} | similarity={score:.3f}]\n{doc.page_content}')
    return '\n\n'.join(parts)


def rag_answer(question: str, k: int = 3, verbose: bool = True) -> str:
    """Full RAG: retrieve → augment prompt → generate answer."""
    # 1. Retrieve
    chunks = retrieve(question, k=k)

    # 2. Build augmented prompt
    context = format_context(chunks)
    user_message = f'CONTEXT:\n{context}\n\nQUESTION: {question}'

    # 3. Generate
    messages = [
        {'role': 'system', 'content': RAG_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_message},
    ]
    answer = ollama_chat(messages)

    if verbose:
        print(f'Q: {question}')
        print(f'Retrieved {len(chunks)} chunks (scores: {[s for _,s in chunks]})')
        print(f'A: {answer}')
        print()

    return answer


# ── Run on 5 questions ────────────────────────────────────────────────────
rag_questions = [
    'What is causing users to be unable to log in?',
    'Which tickets are classified as high priority?',
    'Are there any security-related incidents?',
    'What issues are affecting mobile users?',
    'What low-priority feature requests have been submitted?',
]

print('=' * 70)
print('FULL RAG — 5 QUESTIONS')
print('=' * 70)
for q in rag_questions:
    print('─' * 70)
    rag_answer(q)

---

## Section 6 — Where RAG Fails

RAG dramatically reduces hallucination but does not eliminate it. Understanding
the failure modes is essential for building reliable systems.

### The three main failure modes

| Mode | What happens | Example |
|------|-------------|--------|
| **Out-of-scope question** | No relevant chunks retrieved; model invents an answer anyway | "What is the refund policy?" on a ticket DB with no refund data |
| **Ambiguous question** | Wrong chunks retrieved; confident but irrelevant answer | "What broke last week?" — which of 10 outages? |
| **Answer split across chunks** | Each chunk contains half the answer; neither alone is sufficient | "List all high-priority tickets this month" when the list spans many chunks |

### What to look for
- **Out-of-scope**: the model might say "I don't have that information" (good!) or hallucinate a plausible answer (bad!)
- **Ambiguous**: high similarity scores but the retrieved chunks don't answer the question
- **Split answer**: partial answer; important items missing from the response

In [ ]:
print('=' * 70)
print('RAG FAILURE MODES — DELIBERATE BAD QUERIES')
print('=' * 70)

# ── Failure 1: Question outside the data ──────────────────────────────────
print('\n[FAILURE MODE 1] Out-of-scope question')
print('Expected behaviour: model says it does not have the information.')
print('Risk: model hallucinates a plausible-sounding answer from training data.')
print('─' * 70)
out_of_scope_q = 'What is the company refund and cancellation policy?'
answer1 = rag_answer(out_of_scope_q)

# Retrieve and show what the system actually found
chunks1 = retrieve(out_of_scope_q, k=3)
print(f'Top retrieved chunk (similarity={chunks1[0][1]:.3f}):')
print(chunks1[0][0].page_content[:200])
print('\nWhy it fails: no chunk discusses refunds. Similarity score is low.')
print('The model is forced to guess or (ideally) admit it does not know.')

# ── Failure 2: Ambiguous question ─────────────────────────────────────────
print('\n[FAILURE MODE 2] Ambiguous question')
print('Expected behaviour: answer is vague or picks the wrong incident.')
print('─' * 70)
ambiguous_q = 'What went wrong recently?'
answer2 = rag_answer(ambiguous_q)

chunks2 = retrieve(ambiguous_q, k=3)
print(f'Similarity scores: {[s for _, s in chunks2]}')
print('\nWhy it fails: "recently" has no meaning in our static dataset. The query')
print('is too vague to retrieve a focused chunk. The model picks an arbitrary incident.')
print('Fix: require users to be specific, or add timestamps to filter retrieval.')

# ── Failure 3: Answer split across chunks ─────────────────────────────────
print('\n[FAILURE MODE 3] Answer requires synthesising many chunks')
print('Expected behaviour: incomplete list — some tickets missing.')
print('─' * 70)
synthesis_q = 'Give me a complete list of every high priority ticket in the system.'
answer3 = rag_answer(synthesis_q, k=3)

actual_high_count = len(df[df['priority'] == 'high'])
print(f'\nActual high-priority tickets in dataset: {actual_high_count}')
print('Why it fails: with k=3 we retrieve only 3 chunks. There are', actual_high_count,
      'high-priority tickets.')
print('The model lists what it can see but the list is deeply incomplete.')
print('Fix: increase k, or use a metadata filter (priority=="high") before generation.')

---

## Section 7 — Add Chat History

### Why stateless RAG loses context

The basic `rag_answer()` from Section 5 is stateless: each call is independent.
Ask "Which tickets are high priority?" followed by "Can you summarise the first
one?" and the second call has no idea what "the first one" refers to.

### The solution: prepend recent history

The simplest approach is to include the **last N exchanges** in the prompt
before the retrieved context:

```
[Previous exchange 1 — 2 turns ago]
User: Which tickets are high priority?
Assistant: The high priority tickets are...

[Previous exchange 2 — last turn]
User: What is the most severe one?
Assistant: The most severe is...

CONTEXT: (freshly retrieved chunks for the current question)

QUESTION: Can you tell me who reported it?
```

### Trade-offs

| Approach | Pro | Con |
|----------|-----|-----|
| Last N turns (this notebook) | Simple, no extra infra | Context grows; older turns lost |
| Summarise history | Compact | Loses detail |
| Embed history and retrieve | Theoretically best | Complex |

For most production RAG systems, keeping the last 3–5 turns is sufficient.

### What to look for
- The third question (`"Who reported it?"`) requires knowing what `"it"` refers to — only possible with history
- Token count grows with each turn: print `len(messages)` to see the growing list

In [ ]:
def rag_answer_with_history(
    question: str,
    history: list[dict],
    k: int = 3,
    max_history_turns: int = 2,
) -> tuple[str, list[dict]]:
    """
    RAG with conversation history.

    Parameters
    ----------
    question           : current user question
    history            : list of {'role': ..., 'content': ...} dicts from previous turns
    max_history_turns  : how many previous user+assistant pairs to include

    Returns
    -------
    answer   : model's reply
    history  : updated history list (append current turn)
    """
    # 1. Retrieve fresh context for this question
    chunks  = retrieve(question, k=k)
    context = format_context(chunks)

    # 2. Build history prefix (last max_history_turns exchanges)
    #    Each 'turn' = 1 user message + 1 assistant message = 2 entries
    recent = history[-(max_history_turns * 2):] if history else []

    history_text = ''
    if recent:
        parts = []
        for msg in recent:
            role = 'User' if msg['role'] == 'user' else 'Assistant'
            parts.append(f'{role}: {msg["content"]}')
        history_text = 'PREVIOUS CONVERSATION:\n' + '\n'.join(parts) + '\n\n'

    # 3. Combine history + context + question
    user_content = (
        f'{history_text}'
        f'CONTEXT:\n{context}\n\n'
        f'QUESTION: {question}'
    )

    messages = [
        {'role': 'system', 'content': RAG_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_content},
    ]

    answer = ollama_chat(messages)

    # 4. Update history
    history = history + [
        {'role': 'user',      'content': question},
        {'role': 'assistant', 'content': answer},
    ]

    return answer, history


# ── Test with 3 follow-up questions ───────────────────────────────────────
print('=' * 70)
print('CONVERSATIONAL RAG — 3 FOLLOW-UP QUESTIONS')
print('=' * 70)

conversation = []  # empty history to start

follow_ups = [
    'What authentication-related issues are in the system?',
    'Of those, which one is most severe?',
    'What action should the support team take for that ticket?',
]

for turn, q in enumerate(follow_ups, 1):
    print(f'\n[Turn {turn}] User: {q}')
    answer, conversation = rag_answer_with_history(q, conversation)
    print(f'[Turn {turn}] Assistant: {answer}')
    print(f'           (History length: {len(conversation)} messages)')

print('\n── Full conversation history ──')
for msg in conversation:
    role = 'You      ' if msg['role'] == 'user' else 'Assistant'
    print(f"\n{role}: {msg['content'][:200]}")

---

## ✅ Summary — What You Learned

| # | Concept | Key takeaway |
|---|---------|-------------|
| 1 | RAG problem statement | LLMs hallucinate because they have no access to your data; RAG grounds answers in retrieved facts |
| 2 | Chunking | Split docs into ≤300-char chunks with 50-char overlap; `RecursiveCharacterTextSplitter` respects natural boundaries |
| 3 | Chroma | Local vector store that indexes embeddings in SQLite; no server, no API |
| 4 | Retrieval | Embed query → cosine similarity → top-k chunks; retrieve more than you need then let the model filter |
| 5 | Full RAG | `retrieve → format_context → prompt + context → Ollama → grounded answer` |
| 6 | Failure modes | Out-of-scope, ambiguous, and fragmented-answer queries all degrade RAG quality predictably |
| 7 | Chat history | Prepend last N turns to maintain referential coherence across follow-up questions |

### The architecture you now have

```
support_tickets.csv
       │
       ▼
RecursiveCharacterTextSplitter (chunk_size=300, overlap=50)
       │
       ▼
all-MiniLM-L6-v2 (384-dim embeddings, local, free)
       │
       ▼
ChromaDB (persisted to ./chroma_db)
       │
  similarity_search_with_score
       │
       ▼
rag_answer_with_history(question, history)
       │
       ▼
Ollama llama3 (local, free, grounded)
       │
       ▼
Answer that cites your data ✓
```

### What comes next — Phase 04: Making it Better

The basic RAG pipeline has known weaknesses. Phase 04 addresses them:

| Problem | Phase 04 solution |
|---------|------------------|
| Wrong chunks retrieved | **Re-ranking** with a cross-encoder (BGE-reranker) |
| Incomplete lists | **Metadata filtering** before similarity search |
| Vague queries | **HyDE** (Hypothetical Document Embeddings) — generate a hypothetical answer, embed it, retrieve |
| Context overflow | **Map-reduce** — summarise each chunk separately, then combine |
| Poor chunking | **Semantic chunking** — split on embedding similarity shifts rather than character count |

The RAG pipeline you built here is production-grade for many use cases as-is.
Phase 04 pushes it from "good" to "excellent".